## Código do Resnet


In [1]:
import os
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt 

# ignite
from ignite.engine import Engine, create_supervised_trainer, create_supervised_evaluator
from ignite.handlers import ModelCheckpoint, global_step_from_engine
from ignite.handlers import EarlyStopping

In [2]:
data = "/home/jovyan/DADOS-DIVIDIDOS"
feature_extract= True
batch_size = 128

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomResizedCrop(224),
        # transforms.RandomHorizontalFlip(p=0.5),
        # transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
        transforms.RandomVerticalFlip(),
        transforms.RandomPerspective(distortion_scale=0.2, p=0.5, interpolation=3, fill=0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        # transforms.Resize((224, 224)),
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        # transforms.Resize((224, 224)),
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}


/opt/conda/lib/python3.10/site-packages/torchvision/transforms/transforms.py:768: UserWarning: Argument 'interpolation' of type int is deprecated since 0.13 and will be removed in 0.15. Please use InterpolationMode enum.
  warnings.warn(


In [3]:
image_datasets = {x: datasets.ImageFolder(os.path.join(data, x), data_transforms[x]) for x in ['train', 'val','test']}
# dataloaders_dict = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=batch_size, shuffle=True) for x in ['train', 'val','test']}

In [4]:
dataloaders_dict = {
    'train': torch.utils.data.DataLoader(image_datasets['train'], batch_size=batch_size, shuffle=True),
    'val': torch.utils.data.DataLoader(image_datasets['val'], batch_size=batch_size, shuffle=True),
    'test': torch.utils.data.DataLoader(image_datasets['test'], batch_size=batch_size, shuffle=False)
}

In [5]:
# Extração de features
def set_parameter_requires_grad(model, feature_extracting):
    if feature_extracting:
        for param in model.parameters():
            param.requires_grad = False

## Best trial: Accuracy: 0.9853479853479854 Best hyperparameters: {'dropout1': 0.4720140263972287, 'dropout2': 0.21316165290099465, 'num_neurons_fc1': 256, 'num_neurons_fc2': 256, 'activation': 'ReLU', 'batch_size': 128, 'optimizer': 'SGD', 'lr': 0.0011601308953236539, 'momentum': 0.9895156617003595}


# Código corrigido

In [ ]:
import os
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms
from torch.utils.data import Subset
from sklearn.model_selection import StratifiedKFold
from ignite.metrics import Accuracy, Loss
from ignite.engine import Events, Engine, create_supervised_evaluator
from ignite.handlers import ModelCheckpoint, EarlyStopping
from ignite.contrib.handlers.param_scheduler import LRScheduler
from torch.optim.lr_scheduler import StepLR

# Suas configurações iniciais permanecem as mesmas
dropout_rate1 = 0.4720140263972287
dropout_rate2 = 0.21316165290099465
num_neurons_fc1 = 256 
num_neurons_fc2 = 256
momentum = 0.9895156617003595
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 128
criterion = nn.CrossEntropyLoss()

def create_model():
    model = models.resnet101(pretrained=True)
    
    if feature_extract:
        for param in model.parameters():
            param.requires_grad = False
    
    # Carrega os pesos do modelo pré-treinado
    model_resnet = "/home/jovyan/models/ResNet_101_ImageNet_plant-model-84.pth"
    state_dict = torch.load(model_resnet)
    
    # Remove as camadas fc
    state_dict.pop('fc.weight', None)
    state_dict.pop('fc.bias', None)
    
    # Carrega o state dict modificado
    model.load_state_dict(state_dict, strict=False)
    
    # Redefine a camada fc
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout_rate1),
        nn.Linear(num_features, num_neurons_fc1),
        nn.ReLU(),
        nn.Dropout(p=dropout_rate2),
        nn.Linear(num_neurons_fc1, 2),
        nn.Softmax(dim=1)
    )
    
    return model.to(device)

def train_step(engine, batch):
    model.train()
    inputs, labels = batch[0].to(device), batch[1].to(device)
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    return loss.item()

def validation_step(engine, batch):
    model.eval()
    with torch.no_grad():
        inputs, labels = batch[0].to(device), batch[1].to(device)
        outputs = model(inputs)
        return outputs, labels

n_splits = 10
train_labels = np.array([y for _, y in image_datasets['train']])
skf = StratifiedKFold(n_splits=n_splits, shuffle=True)

all_train_accs_folds = []
all_val_accs_folds = []
all_train_losses_folds = []
all_val_losses_folds = []

val_metrics = {
    "accuracy": Accuracy(),
    "loss": Loss(criterion)
}

def score_function(engine):
    return engine.state.metrics["accuracy"]

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(train_labels)), train_labels)):
    print(f'Fold {fold+1}/{n_splits}')
    
    # Cria um novo modelo para cada fold
    model = create_model()
    
    train_subset = Subset(image_datasets['train'], train_idx)
    val_subset = Subset(image_datasets['train'], val_idx)
    
    train_loader = torch.utils.data.DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_subset, batch_size=batch_size, shuffle=False)

    # Reinicializa o otimizador e scheduler para cada fold
    params_to_update = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.SGD(params_to_update, lr=0.001, momentum=momentum)
    torch_lr_scheduler = StepLR(optimizer, step_size=10, gamma=0.1)
    scheduler = LRScheduler(torch_lr_scheduler)
    criterion = nn.CrossEntropyLoss()
    

    trainer = Engine(train_step)
    evaluator = Engine(validation_step)
    train_evaluator = create_supervised_evaluator(model, metrics=val_metrics, device=device)

    Accuracy().attach(evaluator, 'accuracy')
    Loss(criterion).attach(evaluator, 'loss')
    Accuracy().attach(train_evaluator, 'accuracy')
    Loss(criterion).attach(train_evaluator, 'loss')

    train_accs = []
    val_accs = []
    train_losses = []
    val_losses = []
    
    @trainer.on(Events.STARTED)
    def start_message():
        print(f"Start training fold {fold+1}!")
        
    @trainer.on(Events.EPOCH_COMPLETED)
    def run_train_validation():
        train_evaluator.run(train_loader)

    @trainer.on(Events.EPOCH_COMPLETED)
    def run_validation():
        evaluator.run(val_loader)
        
    @trainer.on(Events.EPOCH_COMPLETED)
    def print_lr():
        print(f"Learning rate atual: {optimizer.param_groups[0]['lr']}")
    
    @train_evaluator.on(Events.COMPLETED)
    def log_train_results():
        metrics = train_evaluator.state.metrics
        train_acc = metrics['accuracy']
        train_loss = metrics['loss']
        train_accs.append(train_acc)
        train_losses.append(train_loss)
        print(f"Fold {fold+1} - Epoch {trainer.state.epoch} - Training Accuracy: {train_acc:.3f}, Loss: {train_loss:.3f}")

    @evaluator.on(Events.COMPLETED)
    def log_validation_results():
        metrics = evaluator.state.metrics
        val_acc = metrics['accuracy']
        val_loss = metrics['loss']
        val_accs.append(val_acc)
        val_losses.append(val_loss)
        print(f"Fold {fold+1} - Epoch: {trainer.state.epoch} - Validation Accuracy: {val_acc:.3f}, Loss: {val_loss:.3f}")

    handler = ModelCheckpoint(
        dirname=f'models_fold_{fold+1}',
        filename_prefix='best',
        n_saved=1,
        create_dir=True,
        score_function=score_function,
        score_name="val_acc",
        require_empty=False
    )
    evaluator.add_event_handler(Events.COMPLETED, handler, {'model': model})

    es_handler = EarlyStopping(patience=50, score_function=score_function, trainer=trainer)
    evaluator.add_event_handler(Events.COMPLETED, es_handler)

    trainer.run(train_loader, max_epochs=500)

    all_train_accs_folds.append(train_accs)
    all_val_accs_folds.append(val_accs)
    all_train_losses_folds.append(train_losses)
    all_val_losses_folds.append(val_losses)

/tmp/ipykernel_119886/2816567346.py:13: DeprecationWarning: /opt/conda/lib/python3.10/site-packages/ignite/contrib/handlers/param_scheduler.py has been moved to /ignite/handlers/param_scheduler.py and will be removed in version 0.6.0.
 Please refer to the documentation for more details.
  from ignite.contrib.handlers.param_scheduler import LRScheduler


Fold 1/10


/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Start training fold 1!
Fold 1 - Epoch 1 - Training Accuracy: 0.512, Loss: 0.690
Fold 1 - Epoch: 1 - Validation Accuracy: 0.525, Loss: 0.690
Learning rate atual: 0.001
Fold 1 - Epoch 2 - Training Accuracy: 0.798, Loss: 0.660
Fold 1 - Epoch: 2 - Validation Accuracy: 0.827, Loss: 0.657
Learning rate atual: 0.001
Fold 1 - Epoch 3 - Training Accuracy: 0.717, Loss: 0.625
Fold 1 - Epoch: 3 - Validation Accuracy: 0.662, Loss: 0.634
Learning rate atual: 0.001
Fold 1 - Epoch 4 - Training Accuracy: 0.706, Loss: 0.596
Fold 1 - Epoch: 4 - Validation Accuracy: 0.719, Loss: 0.600
Learning rate atual: 0.001
Fold 1 - Epoch 5 - Training Accuracy: 0.757, Loss: 0.568
Fold 1 - Epoch: 5 - Validation Accuracy: 0.712, Loss: 0.586
Learning rate atual: 0.001
Fold 1 - Epoch 6 - Training Accuracy: 0.859, Loss: 0.531
Fold 1 - Epoch: 6 - Validation Accuracy: 0.878, Loss: 0.530
Learning rate atual: 0.001
Fold 1 - Epoch 7 - Training Accuracy: 0.894, Loss: 0.489
Fold 1 - Epoch: 7 - Validation Accuracy: 0.899, Loss: 0.

2025-01-14 09:37:56,787 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


...

Fold 7 - Epoch: 110 - Validation Accuracy: 0.964, Loss: 0.350
Learning rate atual: 0.001
Fold 8/10
Start training fold 8!
Fold 8 - Epoch 1 - Training Accuracy: 0.636, Loss: 0.688
Fold 8 - Epoch: 1 - Validation Accuracy: 0.630, Loss: 0.688
Learning rate atual: 0.001
Fold 8 - Epoch 2 - Training Accuracy: 0.721, Loss: 0.664
Fold 8 - Epoch: 2 - Validation Accuracy: 0.746, Loss: 0.659
Learning rate atual: 0.001
Fold 8 - Epoch 3 - Training Accuracy: 0.661, Loss: 0.635
Fold 8 - Epoch: 3 - Validation Accuracy: 0.696, Loss: 0.628
Learning rate atual: 0.001
Fold 8 - Epoch 4 - Training Accuracy: 0.691, Loss: 0.605
Fold 8 - Epoch: 4 - Validation Accuracy: 0.703, Loss: 0.597
Learning rate atual: 0.001
Fold 8 - Epoch 5 - Training Accuracy: 0.778, Loss: 0.576
Fold 8 - Epoch: 5 - Validation Accuracy: 0.768, Loss: 0.571
Learning rate atual: 0.001
Fold 8 - Epoch 6 - Training Accuracy: 0.857, Loss: 0.537
Fold 8 - Epoch: 6 - Validation Accuracy: 0.841, Loss: 0.535
Learning rate atual: 0.001
Fold 8 - Epoch

2025-01-17 06:01:01,360 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Fold 8 - Epoch: 161 - Validation Accuracy: 0.949, Loss: 0.354
Learning rate atual: 0.001
Fold 9/10
Start training fold 9!
Fold 9 - Epoch 1 - Training Accuracy: 0.501, Loss: 0.693
Fold 9 - Epoch: 1 - Validation Accuracy: 0.522, Loss: 0.690
Learning rate atual: 0.001
Fold 9 - Epoch 2 - Training Accuracy: 0.698, Loss: 0.665
Fold 9 - Epoch: 2 - Validation Accuracy: 0.667, Loss: 0.664
Learning rate atual: 0.001
Fold 9 - Epoch 3 - Training Accuracy: 0.599, Loss: 0.636
Fold 9 - Epoch: 3 - Validation Accuracy: 0.558, Loss: 0.638
Learning rate atual: 0.001
Fold 9 - Epoch 4 - Training Accuracy: 0.611, Loss: 0.608
Fold 9 - Epoch: 4 - Validation Accuracy: 0.623, Loss: 0.605
Learning rate atual: 0.001
Fold 9 - Epoch 5 - Training Accuracy: 0.674, Loss: 0.584
Fold 9 - Epoch: 5 - Validation Accuracy: 0.681, Loss: 0.586
Learning rate atual: 0.001
Fold 9 - Epoch 6 - Training Accuracy: 0.809, Loss: 0.550
Fold 9 - Epoch: 6 - Validation Accuracy: 0.790, Loss: 0.558
Learning rate atual: 0.001
Fold 9 - Epoch

2025-01-17 11:54:39,289 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Fold 9 - Epoch: 83 - Validation Accuracy: 0.942, Loss: 0.371
Learning rate atual: 0.001
Fold 10/10
Start training fold 10!
Fold 10 - Epoch 1 - Training Accuracy: 0.589, Loss: 0.679
Fold 10 - Epoch: 1 - Validation Accuracy: 0.630, Loss: 0.677
Learning rate atual: 0.001
Fold 10 - Epoch 2 - Training Accuracy: 0.612, Loss: 0.653
Fold 10 - Epoch: 2 - Validation Accuracy: 0.601, Loss: 0.651
Learning rate atual: 0.001
Fold 10 - Epoch 3 - Training Accuracy: 0.652, Loss: 0.621
Fold 10 - Epoch: 3 - Validation Accuracy: 0.688, Loss: 0.619
Learning rate atual: 0.001
Fold 10 - Epoch 4 - Training Accuracy: 0.772, Loss: 0.590
Fold 10 - Epoch: 4 - Validation Accuracy: 0.768, Loss: 0.594
Learning rate atual: 0.001
Fold 10 - Epoch 5 - Training Accuracy: 0.869, Loss: 0.549
Fold 10 - Epoch: 5 - Validation Accuracy: 0.841, Loss: 0.557
Learning rate atual: 0.001
Fold 10 - Epoch 6 - Training Accuracy: 0.888, Loss: 0.514
Fold 10 - Epoch: 6 - Validation Accuracy: 0.877, Loss: 0.517
Learning rate atual: 0.001
F